In [1]:
print("ok")

ok


In [2]:
%pwd

'/home/rawnakjain/Desktop/Courses/langchain-medibot/research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'/home/rawnakjain/Desktop/Courses/langchain-medibot'

In [6]:
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
def load_pdf_files(directory):
    loader = DirectoryLoader(directory, glob="**/*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

extracted_text = load_pdf_files("data")

In [10]:
len(extracted_text)

637

In [19]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs : List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source", "")
        minimal_docs.append(Document(page_content=doc.page_content, metadata={"source": src}))
    return minimal_docs

filtered_docs = filter_to_minimal_docs(extracted_text)

In [20]:
def text_split(filtered_text):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        separators=["\n\n", "\n", " ", ""]
    )
    texts = text_splitter.split_documents(filtered_text)
    return texts

In [21]:
text_chunk = text_split(filtered_docs)
print(f"Number of chunks: {len(text_chunk)}")

Number of chunks: 3426


In [22]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceBgeEmbeddings(model_name=model_name)

    return embeddings

embeddings = download_embeddings()

/tmp/ipykernel_36270/33983316.py:5: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(model_name=model_name)
/home/rawnakjain/Desktop/Courses/langchain-medibot/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [24]:
print("length of vector is", len(embeddings.embed_query("Hello world")))

length of vector is 384


In [26]:
from dotenv import load_dotenv

load_dotenv()

True

In [27]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY


In [28]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [29]:
pc

In [35]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if index_name not in pc.list_indexes():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [36]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embeddings,
    index_name=index_name,
)

In [37]:
more_data = Document(
    page_content="This is a test document about medical trials.",
    metadata={"source": "test_document.pdf"}
)

docsearch.add_documents(documents=[more_data]
)

['d13d7afa-cc1f-4cc9-9093-b70e47319ddb']

In [38]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [39]:
retrieved_docs = retriever.invoke("Tell me about medical trials.")
retrieved_docs

[Document(id='d13d7afa-cc1f-4cc9-9093-b70e47319ddb', metadata={'source': 'test_document.pdf'}, page_content='This is a test document about medical trials.'),
 Document(id='fe271d5b-21df-46da-a2d3-a2899b1646b9', metadata={'source': 'data/Medical_book.pdf'}, page_content='a cure but it can offer a better quality of life for the patient.\nChemotherapy and radiation therapy have not\nbeen proven effective in the treatment of bile duct cancer.\nPrognosis\nPrognosis depends on the stage and resectability of\nthe tumor. If the patient cannot undergo surgical resec-\ntion, then the survival rate is commonly less than one\nyear. If the tumor is resected, the survival rate improves,\nwith 20% of these patients surviving past five years.\nClinical trials\nStudies of new treatments in patients are known as\nclinical trials. These trials seek to compare the standard\nmethod of care with a new method, or the trials may be try-\ning to establish whether one treatment is more beneficial\nfor certain p

In [42]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",  # or another Gemini model
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

E0000 00:00:1759769558.128614   36270 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [43]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [44]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of growth hormone from the pituitary gland, which leads to increased growth in bone and soft tissue. When this hormonal abnormality occurs in children whose bony growth plates have not yet closed, it is called gigantism, resulting in exceptional height. If the disorder occurs after bone growth has stopped, it is known as acromegaly.
